In [1]:
import numpy as np
from scapy.all import rdpcap
from collections import defaultdict
from scapy.all import IP, TCP, UDP
from scapy.all import Raw

def group_packets_by_session(packet_list):
    sessions = defaultdict(list)
    
    for pkt in packet_list:
        if IP in pkt:
            src_ip = pkt[IP].src
            dst_ip = pkt[IP].dst
            proto = pkt[IP].proto  # 6 for TCP, 17 for UDP
            
            if TCP in pkt:
                src_port = pkt[TCP].sport
                dst_port = pkt[TCP].dport
            elif UDP in pkt:
                src_port = pkt[UDP].sport
                dst_port = pkt[UDP].dport
            else:
                # Skip ICMP or other protocols without ports
                continue
            
            if src_ip < dst_ip:
                session_key = (src_ip, src_port, dst_ip, dst_port, proto)
            else:
                session_key = (dst_ip, dst_port, src_ip, src_port, proto)
            
            sessions[session_key].append(pkt)

    for session_key in sessions:
        sessions[session_key].sort(key=lambda x: x.time)
            
    return sessions

In [2]:
def create_image_from_packet(packet, m):

    if not Raw in packet: return np.zeros((int(m ** 0.5),int(m ** 0.5)), dtype=np.uint8)
    
    payload = packet[Raw].load
    payload_m = payload[0:m] + bytes([0 for i in range(max(0,m - len(payload)))])
    img = np.frombuffer(payload_m, dtype=np.uint8).reshape(int(m ** 0.5),int(m ** 0.5)) 
    return img
    
def create_image_from_session(session, n, m):
    assert int(n ** 0.5) * int(n ** 0.5) == n, "n is not a perfect square"
    assert int(m ** 0.5) * int(m ** 0.5) == m, "m is not a perfect square"

    rn = int(n ** 0.5)
    rm = int(m ** 0.5)

    pkt_images = [create_image_from_packet(packet, m) for packet in session[0:n]] 
    padding_images = [np.zeros((rm,rm), dtype=np.uint8) for extra in range(max(0,n - len(session)))]
    
    image = np.stack(pkt_images + padding_images).reshape(rn,rn,rm,rm).transpose(0,2,1,3)
    return image.reshape(rn * rm ,rn * rm)

In [10]:
from torch.utils.data import Dataset

class SessionImageDataset(Dataset):
    def __init__(self, path_to_pcaps: list, labels_of_pcap: list, n , m):
        super().__init__()
        assert len(path_to_pcaps) == len(labels_of_pcap) ,"length of labels and pcap paths should be same"

        self.data = []
        self.labels = []

        for idx in range(len(path_to_pcaps)):
            packets = rdpcap(path_to_pcaps[idx])
            sessions = group_packets_by_session(packets)

            for session in list(sessions.values()):
                self.data.append(create_image_from_session(session , n, m))
                self.labels.append(labels_of_pcap[idx])

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        return self.data[index],self.labels[index]

In [11]:
data = SessionImageDataset(["archive\\Benign\\Facetime.pcap"],[0], 16, 256)

In [3]:
packets = rdpcap("archive\\Benign\\Facetime.pcap")
sessions = group_packets_by_session(packets)

In [ ]:
list(sessions.values())[0][0]


<Ether  dst=02:1a:c5:02:00:00 src=02:1a:c5:01:00:00 type=IPv4 |<IP  version=4 ihl=5 tos=0x0 len=958 id=39262 flags=DF frag=0 ttl=32 proto=udp chksum=0xf162 src=1.1.58.254 dst=1.2.143.109 |<UDP  sport=16403 dport=16403 len=938 chksum=0x4c54 |<Raw  load=b'\x90h\xa1WW7\xb0\t\xfbN\xb2\xefD\x00\x00\x02(\xf4\xde\xad\x00\x00\x00\x00\xf9\xfe\xf0\xe2\xea=#\xf1\xfc\xcf~\xa6\xf3\xff\x7f\xa3\xcf4\xfb\xe6\xf5Js\xfc\x94\xf9\xc4\xf3n\xde\xefV=\x1e\x8f<\xef\xef\x9fSv\xcc\xf3\xd3?\x1f\x9d\x9b{\xf6\'\x92\xba\xe5m\xff\xd6\x1b\xf5\xfc\xba\xb9I|q7_i\xdb\xf3\xdf$\x93\xff\xdeoq\x9f#\x97)/\x8f\xe7k\xb5?n\xba_\xa3\xff\xdbO\xa9\xb4\xe7\x91\xc5\x92w\xcfC>\xfb\x7f\xd3\xe4\x9e\x8f\xfbh\xa7\xf3\xe2\xea\xf9#\xff=Ow\xbf\x7f\xd7\\\x95\x97\xf4\xe8mSn\xb2="\xb1\xfc\xed\xae\xa7\xef]d\xf5\x977\xebLSz\xce=$\xb1\xf9\xeb\xee\xdbc\x8a\xeb\xaeW\xdf\xebT\xfa\x1a\xff\\\xb2\xb15k\xae\xb8\xef\xf3\'\xa2O\xfb\xebL\xfe\x1c\xffY2NT\xeb\xe6\xde\xe3\xfa$\xa3\xd7\xc7yM\xaaj\xd5{\xb7\xf7U{u\xfb\xd3\x8b\xe4\xbf\xd7\xc7\xeb~\x8a|O_\xb5\xcf?